In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import scipy.stats as stats
from scipy.stats import spearmanr, pearsonr, binomtest

sns.set(style='whitegrid')

In [ ]:
repo_root = Path('/playpen-ssd/wokwen/projects/autoeval_chatbot')
data_path = repo_root / 'ratings' / 'llama' / 'cross_eval_ratings.csv'
df = pd.read_csv(data_path, on_bad_lines='skip')
print('Loaded', df.shape, 'rows')

In [ ]:
def remap_chatbots(df, conv_prefixes=None):
    if conv_prefixes is None:
        conv_prefixes = ['conva','convb','convc']
    rating_cols = {}
    for p in conv_prefixes:
        cand = next((c for c in df.columns if re.fullmatch(p + r'(_|-)?rating', c, flags=re.IGNORECASE) or c.lower().startswith(p + '_rating')), None)
        if cand is None:
            cand = next((c for c in df.columns if c.lower().startswith(p) and 'rating' in c.lower()), None)
        rating_cols[p] = cand
    actual_cols = {p: next((c for c in df.columns if c.lower() == f'actual_{p}'), None) for p in conv_prefixes}
    reason_cols = {p: next((c for c in df.columns if c.lower() in (f'{p}_reason', f'{p}_rating_reason')), None) for p in conv_prefixes}
    generic_reason = next((c for c in df.columns if c.lower() in ('reason','rating_reason','explanation')), None)
    rows = []
    for idx, row in df.iterrows():
        conv_row = {'conv_id': idx}
        for p in conv_prefixes:
            bot_type = None
            if actual_cols.get(p) and pd.notna(row.get(actual_cols[p])):
                bot_type = row.get(actual_cols[p])
            elif p in df.columns and pd.notna(row.get(p)):
                bot_type = row.get(p)
            else:
                bot_type = p
            rc = rating_cols.get(p)
            rating = row.get(rc) if rc in df.columns else row.get(f'{p}_rating') if f'{p}_rating' in df.columns else None
            reason = row.get(reason_cols.get(p)) if reason_cols.get(p) in df.columns else (row.get(generic_reason) if generic_reason else None)
            conv_row[f'{bot_type}_rating'] = rating
            conv_row[f'{bot_type}_reason'] = reason
        rows.append(conv_row)
    return pd.DataFrame(rows)

ratings_wide = remap_chatbots(df)
print('Wide shape:', ratings_wide.shape)

In [ ]:
candidate_patterns = ['_score$', '_criterion$', 'helpful', 'helpfulness', 'accuracy', 'relevance', 'coherence', 'factual', 'correctness', 'fluency']
criteria_cols = []
for c in df.columns:
    lc = c.lower()
    if any(re.search(p, lc) for p in candidate_patterns):
        criteria_cols.append(c)
criteria_cols = [c for c in criteria_cols if not c.lower().endswith('_rating')]
print('Detected criteria columns:', criteria_cols)
if len(criteria_cols) == 0:
    print('No explicit per-criterion columns found. Consider keyword extraction from reason columns.')

In [ ]:
if len(criteria_cols) > 0:
    bot_types = [c.replace('_rating','') for c in ratings_wide.columns if c.endswith('_rating')]
    crit_summary = []
    for bot in bot_types:
        row = {'bot': bot}
        for crit in criteria_cols:
            vals = []
            for slot in ['conva','convb','convc']:
                cand = f
 if not crit.startswith(slot) else crit
                if cand in df.columns:
                    mask = (ratings_wide.get(f
) == ratings_wide.get(f
)) if f
 in ratings_wide.columns else pd.Series([True]*len(ratings_wide))
                    vals.append(df.loc[mask, cand].dropna().astype(float))
            if len(vals) == 0:
                if crit in df.columns:
                    vals.append(df[crit].dropna().astype(float))
            if len(vals) > 0:
                allvals = pd.concat(vals)
                row[crit + '_mean'] = allvals.mean()
                row[crit + '_count'] = allvals.shape[0]
            else:
                row[crit + '_mean'] = np.nan
                row[crit + '_count'] = 0
        crit_summary.append(row)
    crit_df = pd.DataFrame(crit_summary).set_index('bot')
    display(crit_df)
    means = crit_df[[c for c in crit_df.columns if c.endswith('_mean')]]
    if not means.empty:
        plt.figure(figsize=(8, max(2, len(means)*0.6)))
        sns.heatmap(means, annot=True, fmt='.2f', cmap='vlag', center=np.nanmean(means.values))
        plt.title('Per-bot criteria means')
        plt.tight_layout()
        plt.show()
else:
    print('No criteria to summarize.')

### Keyword extraction from free-text reasons (optional)
If structured criteria are missing, use reason columns to extract frequent critique keywords.

In [ ]:
from collections import Counter
import re as _re
reason_cols = [c for c in ratings_wide.columns if c.endswith('_reason')]
print('Found reason columns:', reason_cols)
texts = []
for c in reason_cols:
    texts.extend(ratings_wide[c].dropna().astype(str).tolist())
words = Counter()
for t in texts:
    toks = [w for w in _re.findall(r
, t.lower()) if len(w) > 3]
    words.update(toks)
print('Top tokens:')
print(words.most_common(40))